# Serving smoke test

This notebook starts the shipped stack on Kaggle, vLLM with the base model and the adapter behind the gateway, and checks that it starts and answers. It is a smoke test of the stack, not a measurement of the model, and it has no logic of its own: every step calls `make`.

The steps: install the pinned vLLM, download the base model and the adapter, make the two keys of the test, start `make serve-local` in the background, run `make serve-check`, stop the servers and zip the result file. The check waits for `/readyz`, compares the vLLM version with the pinned one, checks that the base model and the adapter are served, sends a few documents of the dev split through the gateway, and reads the metrics and the audit log. It writes `results/serving/smoke-test.json`, which has no document text in it.

On Kaggle, use one GPU T4 (the server uses the first GPU only) and turn Internet on. Add a secret named `HF_TOKEN` (Add-ons, Secrets): a Hugging Face token that may read the adapter repository, which is gated, so the account must have accepted its terms. Run the notebook with Save Version, Save & Run All (Commit), so the result file is kept in the Output tab. It takes about 20 to 30 minutes, most of it the install, the downloads and the start of vLLM.

In [ ]:
REPO_URL = "https://github.com/mgetech/open-llm-adaptation.git"
REPO_REF = "main"  # a tag or a commit
DOCUMENTS = 6  # documents of the dev split that the check sends through the gateway
WAIT = 1500  # seconds the check waits for /readyz

In [ ]:
import os
import signal
import subprocess
from pathlib import Path


def run(*command, check=True, cwd=None):
    options = dict(stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    with subprocess.Popen(command, encoding="utf-8", errors="replace", **options) as process:
        for line in process.stdout:
            print(line, end="")
    if check and process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


def make(*arguments, check=True):
    run("make", *arguments, check=check)


REPO = Path("/kaggle/working/open-llm-adaptation")
SERVER_LOG = Path("/kaggle/working/serve-local.log")
print("repository:", REPO)

## Python 3.12

Kaggle's default Python may be older, so uv installs its own 3.12.

In [ ]:
run("pip", "install", "--quiet", "uv")
run("uv", "python", "install", "3.12")

## Repository

If the repository is already there, for example after a restart of this cell, it is updated and reused.

In [ ]:
if not REPO.exists():
    run("git", "clone", REPO_URL, str(REPO))
run("git", "-C", str(REPO), "fetch", "--tags", "origin")
run("git", "-C", str(REPO), "checkout", REPO_REF)
os.chdir(REPO)

## Token

The adapter repository is gated. The token comes from the Kaggle secret `HF_TOKEN` and stays in the environment of this session. It is not printed.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

## Project and vLLM

The project is installed from the lock file. vLLM is the version pinned in `configs/models.yaml`.

In [ ]:
make("setup")
make("install-vllm")

## Base model and adapter

`make fetch-weights` downloads the base model at its pinned revision, and `make fetch-adapters` downloads the adapter from the Hub at the pinned commit and checks its hash against `configs/serve.yaml`. After this the servers run offline.

In [ ]:
make("fetch-weights")
make("fetch-adapters")

## Keys of the test

Two keys: one for the platform and one for another platform, to check the 403. `configs/keys.yaml` gets their hashes and the keys themselves go to `logs/`, which git ignores. The command refuses to replace a key file.

In [ ]:
make("serve-check", "PREPARE=1")

## Start the servers

`make serve-local` starts vLLM and the gateway as two processes from `configs/serve.yaml`, in the background. Its output goes to a log file. Starting vLLM takes several minutes, and the check below waits for it.

In [ ]:
log = SERVER_LOG.open("w")
SERVER = subprocess.Popen(
    ["make", "serve-local"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True
)
print("started, process", SERVER.pid)

## Check

Waits for `/readyz`, then runs the checks and writes `results/serving/smoke-test.json`. A failed check is listed with `FAIL`, and the result file is written either way. If the servers stop before they are ready, the check waits until `WAIT` runs out, so look at the log in the next cell.

In [ ]:
make("serve-check", f"DOCUMENTS={DOCUMENTS}", f"WAIT={WAIT}", check=False)

## Log of the servers

The last lines of the log, which show how vLLM started or why it did not.

In [ ]:
run("tail", "-n", "60", str(SERVER_LOG))

## Stop the servers

In [ ]:
if SERVER.poll() is None:
    os.killpg(os.getpgid(SERVER.pid), signal.SIGTERM)
    SERVER.wait(timeout=60)
print("exit code:", SERVER.returncode)

## Result

Copy `results/serving/smoke-test.json` into the same place in the repository. The next cell copies it to `/kaggle/working`, where Kaggle keeps it for a saved version.

In [ ]:
run("cp", "results/serving/smoke-test.json", "/kaggle/working/serving-smoke-test.json")
run("cat", "/kaggle/working/serving-smoke-test.json")